<a href="https://colab.research.google.com/github/Mnzss1133/Visao-Computacional/blob/main/C%C3%B3pia_de_Suspicious_Activity_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 Suspicious Activity Detection in Surveillance Systems Using Deep Learning

**Tema:** Detecção de Atividades Suspeitas em Sistemas de Vigilância com Deep Learning

**Referências-base:**
- Gawande et al. (2024) — YOLOv5 + Motion Feature Map
- Zaidi et al. (2024) — Deep Learning para surveillance (IEEE Access)
- Archana et al. (2026) — GCN baseado em keypoints para reconhecimento de atividades
- Hussain et al. (2024) — Framework de biometria comportamental com IA
- Naseer et al. (2022) — Detecção com Deep Learning e Raspberry Pi
- Mishra et al. (2025) — Framework inteligente para atividades criminosas suspeitas
- Berardini et al. (2025) — IA de borda + super-resolução para detecção de armas
- Shanthi & Manjula (2025) — YOLOv8 para prevenção de crimes

---
> **Pipeline:** Input de imagem → YOLOv8 (detecção de pessoas) → Extração de keypoints (pose) → Análise de comportamento → Classificação de atividade suspeita → Alerta


## 📦 Célula 1 — Instalação de Dependências

In [ ]:
!pip install ultralytics opencv-python-headless matplotlib Pillow numpy -q

import sys
print(f"Python {sys.version}")
print("✅ Dependências instaladas com sucesso!")


Python 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
✅ Dependências instaladas com sucesso!


In [ ]:
import zipfile

zip_path = "/content/gun detection.v1i.yolov8.zip"
extract_path = "/content/gun-detection"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Dataset descompactado!")

✅ Dataset descompactado!


In [ ]:
!cat /content/gun-detection/data.yaml

train: ../train/images
val: ../valid/images
test: ../test/images

nc: 1
names: ['gun']

roboflow:
  workspace: workspace-1qko2
  project: gun-detection-ghlzd
  version: 1
  license: Private
  url: https://app.roboflow.com/workspace-1qko2/gun-detection-ghlzd/1

In [ ]:
with open("/content/gun-detection/data.yaml", "w") as f:
    f.write("""
train: /content/gun-detection/train/images
val: /content/gun-detection/valid/images
test: /content/gun-detection/test/images

nc: 1
names: ['gun']
""")

In [ ]:
import os

print(os.path.exists("/content/gun-detection/train/images"))
print(os.path.exists("/content/gun-detection/valid/images"))

True
True


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

model.train(
    data="/content/gun-detection/data.yaml",
    epochs=5,
    imgsz=640
)

Ultralytics 8.4.41 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/gun-detection/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=5, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-4, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=

## 📚 Célula 2 — Importações e Configuração

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from ultralytics import YOLO
import urllib.request
import os
import warnings
warnings.filterwarnings("ignore")

# ── Configurações de exibição ──────────────────────────────────────────────
plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["font.family"] = "DejaVu Sans"

print("✅ Imports concluídos!")


## 🤖 Célula 3 — Carregamento dos Modelos

Carregamos dois modelos YOLOv8:
- **YOLOv8n** → detecção de objetos/pessoas/armas (baseado em Shanthi & Manjula 2025; Berardini et al. 2025)
- **YOLOv8n-pose** → estimativa de pose/keypoints (baseado em Archana et al. 2026; Hussain et al. 2024)


In [ ]:
from ultralytics import YOLO
import os
from huggingface_hub import hf_hub_download

# ── CONFIGURAÇÕES ──────────────────────────────────────────
WEAPON_MODEL_PATH = "weapon_model.pt"
# Tentando um repositório alternativo que costuma estar ativo
REPO_ID_ALT = "fcakyon/yolov8n-weapons"

# ── DOWNLOAD COM TRATAMENTO DE ERRO ────────────────────────
model_weapon_gun = None

if not os.path.exists(WEAPON_MODEL_PATH):
    print("Iniciando busca pelo modelo de armas...")
    try:
        # Tenta baixar o modelo alternativo
        path = hf_hub_download(repo_id=REPO_ID_ALT, filename="best.pt")
        import shutil
        shutil.copy(path, WEAPON_MODEL_PATH)
        print(f"✅ Modelo baixado com sucesso: {WEAPON_MODEL_PATH}")
    except Exception as e:
        print(f"⚠️ Não foi possível baixar o modelo especializado. Erro: {e}")
        print("💡 O sistema operará apenas com o modo Fallback (Classes COCO).")

# Só carrega se o arquivo realmente existir
if os.path.exists(WEAPON_MODEL_PATH):
    model_weapon_gun = YOLO(WEAPON_MODEL_PATH)
else:
    model_weapon_gun = None

# Modelos base sempre carregam (são nativos da Ultralytics)
model_det = YOLO("yolov8n.pt")

# ── CLASSE DETECTOR (AJUSTADA) ─────────────────────────────
class WeaponDetectorFallback:
    WEAPON_CLASSES_COCO = {
        43: "knife",
        76: "scissors",
        39: "bottle",
        38: "baseball bat",
    }

    def __init__(self, model_coco, model_gun):
        self.model_coco = model_coco
        self.model_gun = model_gun

    def detect(self, img, conf=0.25):
        weapons = []

        # 1. Detecção COCO (Sempre tenta)
        results_coco = self.model_coco(img, conf=conf, verbose=False)[0]
        for box in results_coco.boxes:
            cls_id = int(box.cls[0])
            if cls_id in self.WEAPON_CLASSES_COCO:
                weapons.append({
                    'label': f"{self.WEAPON_CLASSES_COCO[cls_id]} (COCO)",
                    'conf':  float(box.conf[0]),
                    'box':   box.xyxy[0].cpu().numpy()
                })

        # 2. Detecção Especializada (Só se o modelo foi carregado)
        if self.model_gun is not None:
            results_gun = self.model_gun(img, conf=conf, verbose=False)[0]
            for box in results_gun.boxes:
                cls_name = self.model_gun.names[int(box.cls[0])].lower()
                weapons.append({
                    'label': f"{cls_name} (Especializado)",
                    'conf':  float(box.conf[0]),
                    'box':   box.xyxy[0].cpu().numpy()
                })

        return weapons

# Inicialização
model_weapon = WeaponDetectorFallback(model_det, model_weapon_gun)

print("-" * 30)
if model_weapon_gun:
    print("✅ SISTEMA COMPLETO: Modelos COCO e Especializado ativos.")
else:
    print("⚠️ SISTEMA PARCIAL: Apenas detecção via COCO ativa.")

## 🧠 Célula 4 — Módulo de Análise de Comportamento

Implementa a lógica de classificação de atividades suspeitas com base em:
- **Análise de pose** → ângulos entre articulações (joelhos, cotovelos, ombros)
- **Proporção corporal** → razão altura/largura da bounding box
- **Contexto de objetos** → presença de armas, facas etc.

Baseado em: Gawande et al. (2024), Zaidi et al. (2024), Mishra et al. (2025)


In [ ]:
class BehaviorAnalyzer:
    """
    Analisador de comportamento baseado em pose e contexto de objetos.

    Referências:
    - Gawande et al. (2024): YOLOv5 + Motion Feature Map
    - Hussain et al. (2024): AI-Driven Behavior Biometrics Framework
    - Archana et al. (2026): Keypoint-Based GCN para surveillance
    """

    # Índices dos keypoints COCO (YOLOv8-pose)
    KP = {
        "nose": 0, "l_eye": 1, "r_eye": 2, "l_ear": 3, "r_ear": 4,
        "l_shoulder": 5, "r_shoulder": 6, "l_elbow": 7, "r_elbow": 8,
        "l_wrist": 9, "r_wrist": 10, "l_hip": 11, "r_hip": 12,
        "l_knee": 13, "r_knee": 14, "l_ankle": 15, "r_ankle": 16,
    }

    # Classes COCO consideradas suspeitas/armas
    WEAPON_CLASSES = {43: "knife", 76: "scissors", 44: "spoon"}
    WEAPON_CLASSES_NAMES = {"knife", "scissors", "gun", "pistol", "rifle"}

    def __init__(self, conf_threshold: float = 0.5):
        self.conf_threshold = conf_threshold

    # ── Utilitários de geometria ──────────────────────────────────────────
    @staticmethod
    def _angle(a, b, c):
        """Ângulo em graus no ponto b formado pelos vetores ba e bc."""
        a, b, c = np.array(a[:2]), np.array(b[:2]), np.array(c[:2])
        ba, bc = a - b, c - b
        cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
        return np.degrees(np.arccos(np.clip(cos, -1, 1)))

    @staticmethod
    def _visible(kps, *indices, thr=0.3):
        """Verifica se todos os keypoints têm confiança acima do limiar."""
        return all(kps[i][2] >= thr for i in indices)

    # ── Análise de pose ───────────────────────────────────────────────────
    def analyze_pose(self, keypoints):
        """
        Retorna lista de comportamentos suspeitos detectados via pose.
        Baseado em Archana et al. (2026) e Hussain et al. (2024).
        """
        flags = []
        if keypoints is None or len(keypoints) == 0:
            return flags

        kps = keypoints  # shape: (17, 3) → x, y, conf

        KP = self.KP

        # 1. Mãos levantadas acima da cabeça (rendição / ameaça)
        if self._visible(kps, KP["l_wrist"], KP["nose"]):
            if kps[KP["l_wrist"]][1] < kps[KP["nose"]][1]:
                flags.append("mão esquerda levantada")
        if self._visible(kps, KP["r_wrist"], KP["nose"]):
            if kps[KP["r_wrist"]][1] < kps[KP["nose"]][1]:
                flags.append("mão direita levantada")

        # 2. Postura agachada (joelhos muito dobrados)
        for side, knee, hip, ankle in [
            ("esquerdo", KP["l_knee"], KP["l_hip"], KP["l_ankle"]),
            ("direito",  KP["r_knee"], KP["r_hip"], KP["r_ankle"]),
        ]:
            if self._visible(kps, knee, hip, ankle):
                ang = self._angle(kps[hip], kps[knee], kps[ankle])
                if ang < 100:
                    flags.append(f"agachado (joelho {side}: {ang:.0f}°)")

        # 3. Inclinação acentuada do tronco (queda / fuga)
        if self._visible(kps, KP["l_shoulder"], KP["r_shoulder"], KP["l_hip"], KP["r_hip"]):
            shoulder_mid_y = (kps[KP["l_shoulder"]][1] + kps[KP["r_shoulder"]][1]) / 2
            hip_mid_y      = (kps[KP["l_hip"]][1]      + kps[KP["r_hip"]][1])      / 2
            shoulder_mid_x = (kps[KP["l_shoulder"]][0] + kps[KP["r_shoulder"]][0]) / 2
            hip_mid_x      = (kps[KP["l_hip"]][0]      + kps[KP["r_hip"]][0])      / 2
            dx = abs(shoulder_mid_x - hip_mid_x)
            dy = abs(shoulder_mid_y - hip_mid_y) + 1e-6
            if dx / dy > 0.6:
                flags.append(f"tronco inclinado (dx/dy={dx/dy:.2f})")

        # 4. Braços estendidos para frente (possível ameaça/disparo)
        for side, shoulder, elbow, wrist in [
            ("esq.", KP["l_shoulder"], KP["l_elbow"], KP["l_wrist"]),
            ("dir.", KP["r_shoulder"], KP["r_elbow"], KP["r_wrist"]),
        ]:
            if self._visible(kps, shoulder, elbow, wrist):
                ang = self._angle(kps[shoulder], kps[elbow], kps[wrist])
                if ang > 155:
                    flags.append(f"braço {side} estendido ({ang:.0f}°)")

        return flags

    # ── Análise de proporção corporal (bounding box) ──────────────────────
    def analyze_bbox_ratio(self, box):
        """
        Razão largura/altura da bbox.
        Pessoa deitada → razão alta (suspeito).
        Baseado em Naseer et al. (2022) e Mishra et al. (2025).
        """
        x1, y1, x2, y2 = box
        w, h = x2 - x1, y2 - y1
        if h == 0:
            return []
        ratio = w / h
        if ratio > 1.4:
            return [f"pessoa deitada/caída (w/h={ratio:.2f})"]
        return []

    # ── Classificação final ───────────────────────────────────────────────
    def classify(self, pose_flags, bbox_flags, weapon_detected=False):
      score = 0
      label = "normal"

      flags = pose_flags + bbox_flags

      # 🔥 REGRA 1 — ARMA + BRAÇO ESTENDIDO = AMEAÇA
      if weapon_detected and any("estendido" in f for f in pose_flags):
          return "CRÍTICO", "critico", 10

      # 🔥 REGRA 2 — ARMA + MÃO LEVANTADA = RENDIÇÃO
      if weapon_detected and any("levantada" in f for f in pose_flags):
          return "SUSPEITO", "suspeito", 3

      # 🔥 REGRA 3 — ARMA + CORPO INCLINADO = MOVIMENTO AGRESSIVO
      if weapon_detected and any("inclinado" in f for f in pose_flags):
          return "ALTO RISCO", "alto_risco", 5

      # 🔥 REGRA 4 — SÓ ARMA
      if weapon_detected:
          return "ALTO RISCO", "alto_risco", 5

      # fallback original
      for f in flags:
          if any(k in f for k in ["levantada", "estendido"]):
              score += 2
          elif any(k in f for k in ["agachado", "inclinado", "deitado"]):
              score += 2
          else:
              score += 1

      if score == 0:
          return "NORMAL", "normal", score
      elif score <= 2:
          return "SUSPEITO", "suspeito", score
      elif score <= 4:
          return "ALTO RISCO", "alto_risco", score
      else:
          return "CRÍTICO", "critico", score

print("✅ BehaviorAnalyzer definido!")
analyzer = BehaviorAnalyzer()


## 🎨 Célula 5 — Funções de Visualização

In [ ]:
RISK_COLORS = {
    "normal":     ("#2ecc71", "white"),
    "suspeito":   ("#f39c12", "white"),
    "alto_risco": ("#e67e22", "white"),
    "critico":    ("#e74c3c", "white"),
}

POSE_CONNECTIONS = [
    (5, 6), (5, 7), (7, 9), (6, 8), (8, 10),
    (5, 11), (6, 12), (11, 12), (11, 13), (13, 15),
    (12, 14), (14, 16), (0, 5), (0, 6),
]

def draw_skeleton(ax, keypoints, color="#00d4ff"):
    """Desenha esqueleto sobre a imagem — estilo Archana et al. (2026)."""
    if keypoints is None:
        return
    kps = keypoints
    for i, j in POSE_CONNECTIONS:
        if kps[i][2] > 0.3 and kps[j][2] > 0.3:
            ax.plot([kps[i][0], kps[j][0]], [kps[i][1], kps[j][1]],
                    color=color, linewidth=1.8, alpha=0.85)
    for k in kps:
        if k[2] > 0.3:
            ax.scatter(k[0], k[1], c=color, s=18, zorder=5, edgecolors="white", linewidths=0.5)

def draw_result(image_bgr, detections, title="Análise"):
    """
    Renderiza imagem com bounding boxes, esqueleto e rótulos de risco.
    """
    img_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    fig.patch.set_facecolor("#1a1a2e")

    for ax in axes:
        ax.imshow(img_rgb)
        ax.set_xlim(0, w); ax.set_ylim(h, 0)
        ax.axis("off")

    ax_raw, ax_ann = axes
    ax_raw.set_title("Imagem Original", color="white", pad=8)
    ax_ann.set_title(f"Detecção — {title}", color="white", pad=8)

    summary_lines = []

    for d in detections:
        box   = d["box"]
        label = d["label"]
        level = d["level"]
        flags = d["flags"]
        kps   = d.get("keypoints")
        pid   = d.get("pid", "")

        color_hex, txt_color = RISK_COLORS.get(level, ("#888", "white"))

        x1, y1, x2, y2 = box
        bw, bh = x2 - x1, y2 - y1

        rect = patches.FancyBboxPatch(
            (x1, y1), bw, bh,
            linewidth=2.2, edgecolor=color_hex,
            facecolor="none",
            boxstyle="round,pad=1"
        )
        ax_ann.add_patch(rect)

        tag = f"{label}  [{pid}]" if pid else label
        ax_ann.text(
            x1, y1 - 6, tag,
            color=txt_color, fontsize=8.5, fontweight="bold",
            bbox=dict(facecolor=color_hex, alpha=0.88, pad=2, edgecolor="none")
        )

        # Flags de comportamento
        for fi, flag in enumerate(flags[:3]):
            ax_ann.text(x1 + 4, y2 + 12 + fi * 13, f"• {flag}",
                        color="#ffdd57", fontsize=6.5, alpha=0.9)

        draw_skeleton(ax_ann, kps, color=color_hex)
        summary_lines.append(f"  [{pid}] {label}: {', '.join(flags) if flags else 'sem anomalias'}")

    fig.suptitle(
        "Suspicious Activity Detection — Deep Learning Pipeline",
        color="white", fontsize=13, fontweight="bold", y=1.01
    )

    legend_patches = [
        patches.Patch(color=c, label=lbl)
        for lbl, (c, _) in zip(
            ["Normal", "Suspeito", "Alto Risco", "Crítico"],
            RISK_COLORS.values()
        )
    ]
    ax_ann.legend(handles=legend_patches, loc="upper right",
                  fontsize=7.5, framealpha=0.7,
                  facecolor="#1a1a2e", labelcolor="white")

    plt.tight_layout()
    plt.show()

    print("\n📋 Resumo da análise:")
    for line in summary_lines:
        print(line)

print("✅ Funções de visualização definidas!")


## 🔬 Célula 6 — Pipeline Principal de Detecção

Pipeline completo:
1. Detecção de pessoas com YOLOv8 (Gawande et al. 2024)
2. Detecção de armas/objetos perigosos (Shanthi & Manjula 2025; Berardini et al. 2025)
3. Estimativa de pose corporal (Archana et al. 2026)
4. Análise comportamental (Hussain et al. 2024; Zaidi et al. 2024)
5. Classificação de risco e alerta (Mishra et al. 2025; Naseer et al. 2022)


In [ ]:
## 🔬 Célula 6 — Pipeline Principal de Detecção (CORRIGIDA)

def check_overlap(box1, box2):
    """ Verifica se duas caixas (x1, y1, x2, y2) se sobrepõem """
    x1_max = max(box1[0], box2[0])
    y1_max = max(box1[1], box2[1])
    x2_min = min(box1[2], box2[2])
    y2_min = min(box1[3], box2[3])
    return x1_max < x2_min and y1_max < y2_min

def is_weapon_near_hand(kps, weapon_box, threshold=100):
    if kps is None: return False
    wx1, wy1, wx2, wy2 = weapon_box
    wx, wy = (wx1 + wx2)/2, (wy1 + wy2)/2
    for idx in [9, 10]: # pulsos
        if kps[idx][2] > 0.3:
            hx, hy = kps[idx][:2]
            dist = np.linalg.norm([hx - wx, hy - wy])
            if dist < threshold: return True
    return False

def detect_suspicious_activity(input_data, conf=0.15, title=""):
    # Carregamento da imagem (suporta caminho ou array numpy)
    if isinstance(input_data, str):
        img = cv2.imread(input_data)
    else:
        img = input_data

    if img is None:
        print("❌ Erro ao carregar imagem.")
        return []

    detections = []

    # 1. Detecção de armas (Modelo especializado + COCO Fallback)
    armas_encontradas = model_weapon.detect(img, conf=conf)

    # 2. Detecção de pessoas e poses
    res_pose = model_pose(img, conf=conf, verbose=False)[0]

    # Se não houver pessoas, ainda queremos mostrar se há armas isoladas
    # Mas o pipeline foca em pessoas, então vamos iterar por elas
    if res_pose.boxes is not None:
        for pid, person in enumerate(res_pose.boxes):
            if int(person.cls[0]) != 0: continue # pula se não for pessoa

            p_box = person.xyxy[0].cpu().numpy()
            conf_val = float(person.conf[0])

            kps = None
            if res_pose.keypoints is not None:
                kps = res_pose.keypoints.data[pid].cpu().numpy()

            # Análise de comportamento
            pose_flags = analyzer.analyze_pose(kps)
            bbox_flags = analyzer.analyze_bbox_ratio(p_box)

            # Verificação de armas próximas a esta pessoa específica
            weapon_near = False
            for arma in armas_encontradas:
                # Checa se a arma está perto da mão ou dentro da box da pessoa
                if is_weapon_near_hand(kps, arma['box']) or check_overlap(p_box, arma['box']):
                    weapon_near = True
                    pose_flags.append(f"⚠️ portando {arma['label']}")

            # Classificação final de risco
            label, level, score = analyzer.classify(pose_flags, bbox_flags, weapon_near)

            detections.append({
                "pid": f"P{pid+1}",
                "box": p_box,
                "conf": conf_val,
                "keypoints": kps,
                "flags": pose_flags + bbox_flags,
                "label": label,
                "level": level,
                "score": score
            })

    # Imprime debug das armas no console
    print(f"DEBUG: Armas detectadas no frame: {len(armas_encontradas)}")
    for a in armas_encontradas:
        print(f"  - {a['label']} ({a['conf']:.2f})")

    # Visualização
    draw_result(img, detections, title=title or "Análise")
    return detections

print("✅ Pipeline corrigido e unificado!")

## 📸 Célula 7 — Download de Imagens de Teste

### Como Mudar as Imagens de Teste

Você pode facilmente trocar as imagens de teste editando a célula de código abaixo (`Célula 7`).

Existem duas maneiras principais de fazer isso:

1.  **Atualizar o dicionário `test_images`:** Edite as entradas no dicionário `test_images` para incluir URLs de suas próprias imagens. Cada entrada deve ter o nome do arquivo como chave e a URL como valor.
2.  **Alterar `test_url`:** Modifique a variável `test_url` para apontar para uma única imagem que você deseja usar como o principal teste (como `test_bus.jpg` foi usada).

Lembre-se de executar a célula novamente após fazer as alterações para baixar as novas imagens.

In [ ]:
import urllib.request

test_images = {
    "pessoas_rua.jpg":
        "https://upload.wikimedia.org/wikipedia/commons/thumb/b/be/Pedestrians_crossing_Shibuya_Crossing.jpg/1280px-Pedestrians_crossing_Shibuya_Crossing.jpg",
    "pessoa_sozinha.jpg":
        "https://upload.wikimedia.org/wikipedia/commons/thumb/1/14/Gatto_europeo4.jpg/200px-Gatto_europeo4.jpg",
}

# Imagem mais confiável para teste de pose
test_url = "https://ultralytics.com/images/bus.jpg"
urllib.request.urlretrieve(test_url, "test_bus.jpg")
print("✅ Imagem de teste baixada: test_bus.jpg")

# Para usar sua própria imagem, faça upload no Colab:
# from google.colab import files
# uploaded = files.upload()
# img_path = list(uploaded.keys())[0]


In [ ]:
import urllib.request
import shutil
import os
caminho_original = "/content/whatsapp-image-2017-10-27-at-14.13.12.webp"
nome_destino = "multidao.jpg"

if os.path.exists(caminho_original):
    shutil.copy(caminho_original, nome_destino)
    print(f"✅ Sucesso! O arquivo local foi copiado para: {nome_destino}")
else:
    print("❌ Erro: O arquivo original não foi encontrado na pasta /content/")

In [ ]:
print(dataset.location)

## ▶️ Célula 8 — Executar Detecção na Imagem de Teste

In [ ]:
print("=" * 60)
print("🔍 SUSPICIOUS ACTIVITY DETECTION PIPELINE")
print("=" * 60)

results = detect_suspicious_activity(
    "multidao.jpg",
    conf=0.05,
    title="Teste — Imagem de Ônibus"
)

print(f"\n✅ {len(results)} pessoa(s) analisada(s).")


## 📂 Célula 9 — Usar Sua Própria Imagem (upload)

In [ ]:
# ─── Descomente o bloco abaixo para fazer upload de uma imagem no Colab ───

# from google.colab import files
# uploaded = files.upload()
# img_path = list(uploaded.keys())[0]
# print(f"Imagem carregada: {img_path}")

# results = detect_suspicious_activity(img_path, conf=0.30, title=img_path)

# ─── Ou carregue via URL ───────────────────────────────────────────────────

# import urllib.request
# url = "https://URL_DA_SUA_IMAGEM.jpg"
# urllib.request.urlretrieve(url, "custom.jpg")
# results = detect_suspicious_activity("custom.jpg", conf=0.30, title="Custom")
print("ℹ️  Descomente uma das opções acima para usar sua própria imagem.")


## 📊 Célula 9.1 — Demonstração de Cálculo de Métricas (Hipótetico)

Para calcular métricas de desempenho como acurácia, precisão e recall, é essencial ter um conjunto de dados com o **'Ground Truth' (Verdade Fundamental)**. Isso significa que, para cada detecção ou classificação feita pelo modelo, precisamos saber qual era a resposta "correta" esperada.

Como o pipeline atual foca em inferência e os dados de teste (`test_bus.jpg` e os cenários sintéticos) não possuem um 'ground truth' formal, a célula abaixo apresenta uma **demonstração hipotética** de como você poderia estruturar o cálculo de métricas. Ela usa os resultados da imagem `test_bus.jpg` e um 'ground truth' **fictício** para ilustrar o processo.

In [ ]:
def calculate_mock_metrics(predictions, ground_truth):
    """
    Demonstração hipotética de cálculo de métricas básicas.
    Assume que ground_truth é uma lista de dicionários com 'pid' e 'level' esperados.
    """
    if not ground_truth:
        print("Ground truth vazio. Nenhuma métrica para calcular.")
        return {}

    correct_classifications = 0
    total_people = len(ground_truth)

    print("--- Comparando Previsões com Ground Truth --- ")

    # Mapeia ground truth por PID para fácil acesso
    gt_map = {item['pid']: item for item in ground_truth}

    for pred in predictions:
        pid = pred['pid']
        pred_level = pred['level']

        if pid in gt_map:
            gt_level = gt_map[pid]['level']
            # Convertendo gt_level para minúsculas para correspondência
            print(f"  Pessoa {pid}: Previsto='{pred_level}', GroundTruth='{gt_level}'")
            if pred_level == gt_level.lower(): # Correção aqui!
                correct_classifications += 1
        else:
            print(f"  Pessoa {pid}: Previsão '{pred_level}', sem Ground Truth correspondente.")

    if total_people > 0:
        accuracy = correct_classifications / total_people
        print(f"\n--- Resultados Hipotéticos ---")
        print(f"  Classificações Corretas: {correct_classifications}/{total_people}")
        print(f"  Acurácia Simples: {accuracy:.2f}")
        return {"simple_accuracy": accuracy}
    else:
        print("Nenhuma pessoa para avaliar no Ground Truth.")
        return {}

# ----------------------------------------------------------------------
# Exemplo de uso com resultados da imagem 'test_bus.jpg'
# Nota: O 'ground_truth_mock' é FICTÍCIO e criado APENAS para esta demonstração.
# ----------------------------------------------------------------------

# Recupere os resultados da última execução na test_bus.jpg (Célula 8)
# Certifique-se de que 'results' contém os resultados de 'test_bus.jpg' se você o executou.
# Se não, execute a Célula 8 primeiro para popular a variável `results`.

# Criar um Ground Truth Fictício para a imagem 'test_bus.jpg'
ground_truth_mock = [
    {"pid": "P1", "level": "NORMAL"},
    {"pid": "P2", "level": "NORMAL"},
    {"pid": "P3", "level": "SUSPEITO"},
    {"pid": "P4", "level": "NORMAL"}
]

print("\nExecutando pipeline para obter previsões da test_bus.jpg novamente...")
current_predictions = detect_suspicious_activity("multidao.jpg", conf=0.35, title="Teste para Métricas")

print("\nIniciando cálculo de métricas mock...")
metrics = calculate_mock_metrics(current_predictions, ground_truth_mock)
print("\n✅ Cálculo de métricas mock concluído!")


## 📊 Célula 10 — Análise em Lote (múltiplas imagens)

In [ ]:
import glob

def batch_analyze(image_paths: list, conf: float = 0.35):
    """
    Processa múltiplas imagens e retorna um relatório consolidado.
    Baseado em Zaidi et al. (2024) e Mishra et al. (2025).
    """
    report = []

    for path in image_paths:
        print(f"\n{'─'*50}")
        print(f"📸 Processando: {path}")
        try:
            results = detect_suspicious_activity(path, conf=conf, title=os.path.basename(path))
            for r in results:
                report.append({
                    "imagem": path,
                    "pessoa": r["pid"],
                    "nivel":  r["label"],
                    "score":  r["score"],
                    "flags":  r["flags"],
                })
        except Exception as e:
            print(f"  ❌ Erro: {e}")

    # Resumo
    print(f"\n{'='*50}")
    print("📋 RELATÓRIO CONSOLIDADO")
    print(f"{'='*50}")
    criticos  = [r for r in report if r["nivel"] == "CRÍTICO"]
    altos     = [r for r in report if r["nivel"] == "ALTO RISCO"]
    suspeitos = [r for r in report if r["nivel"] == "SUSPEITO"]
    normais   = [r for r in report if r["nivel"] == "NORMAL"]

    print(f"  🔴 Críticos   : {len(criticos)}")
    print(f"  🟠 Alto Risco : {len(altos)}")
    print(f"  🟡 Suspeitos  : {len(suspeitos)}")
    print(f"  🟢 Normais    : {len(normais)}")
    print(f"  Total pessoas analisadas: {len(report)}")
    return report

# Exemplo com a imagem de teste
relatorio = batch_analyze(["multidao.jpg"])


## 🧪 Célula 11 — Cenário Sintético (sem câmera real)

In [ ]:
def create_synthetic_frame(scenario: str = "normal"):
    """
    Gera um frame sintético com padrão de pixels para simular cenários.
    Útil para testar o pipeline sem imagens reais.
    """
    h, w = 480, 640
    frame = np.ones((h, w, 3), dtype=np.uint8) * 40  # fundo escuro

    if scenario == "crowded":
        # Simula multidão (vários retângulos representando pessoas)
        for i in range(5):
            x = 80 + i * 110
            cv2.rectangle(frame, (x, 150), (x + 50, 420), (180, 160, 140), -1)
        label = "Cena com Multidão"
    elif scenario == "fallen":
        # Pessoa deitada (bbox horizontal)
        cv2.rectangle(frame, (100, 280), (540, 360), (160, 140, 120), -1)
        label = "Possível Queda/Pessoa Deitada"
    else:
        # Cena normal com uma pessoa em pé
        cv2.rectangle(frame, (270, 100), (370, 420), (170, 150, 130), -1)
        label = "Cena Normal"

    # Adiciona grid de fundo (simula chão)
    for y in range(0, h, 40):
        cv2.line(frame, (0, y), (w, y), (60, 60, 60), 1)
    for x in range(0, w, 40):
        cv2.line(frame, (x, 0), (x, h), (60, 60, 60), 1)

    cv2.putText(frame, label, (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)
    return frame, label

for scenario in ["normal", "crowded", "fallen"]:
    frame, label = create_synthetic_frame(scenario)
    print(f"\n{'─'*50}")
    print(f"🎬 Cenário: {label}")
    detect_suspicious_activity(frame, conf=0.20, title=label)


## 📚 Célula 12 — Referências Bibliográficas

| # | Referência | Contribuição neste código |
|---|-----------|--------------------------|
| [1] | Gawande et al. (2024). *YOLOv5 + Motion Feature Map*. Artificial Intelligence Review. | Base da detecção com YOLO |
| [2] | Zaidi et al. (2024). *Suspicious Human Activity Recognition*. IEEE Access. | Classificação de atividades suspeitas |
| [3] | Archana et al. (2026). *Keypoint-Based GCN for Surveillance*. Discover Applied Sciences. | Análise de keypoints e pose |
| [4] | Hussain et al. (2024). *AI-Driven Behavior Biometrics*. Engineering Applications of AI. | Framework de análise comportamental |
| [5] | Naseer et al. (2022). *Deep Learning + Raspberry Pi*. Sensors. | Pipeline para sistemas embarcados |
| [6] | Mishra et al. (2025). *Intelligent Framework for Criminal Activity*. JRIST. | Classificação de risco e alertas |
| [7] | Berardini et al. (2025). *Edge AI + Super-Resolution*. Engineering Applications of AI. | Detecção de armas em vídeo |
| [8] | Shanthi & Manjula (2025). *YOLOv8 for Crime Prevention*. Scientific Reports. | Modelo YOLOv8 para detecção |

---
> **Nota:** Este notebook implementa um pipeline de pesquisa acadêmica. Para uso em produção, considere datasets rotulados específicos (ex.: UCF-Crime, VIRAT, ShanghaiTech) e fine-tuning dos modelos.
